## 1) โหลด speech VAD (จาก CSV ที่เราเพิ่งสร้าง)

In [2]:
import pandas as pd
import numpy as np

speech_csv = "emotion_results_wavlm_all.csv"
df_speech = pd.read_csv(speech_csv)

df_speech

,filename,dialogue_id,utterance_id,arousal,valence,dominance
0,dialogue_3_utterance_1.wav,3,1,0.557219,0.181282,0.630059
1,dialogue_3_utterance_2.wav,3,2,0.727785,0.207596,0.751352
2,dialogue_3_utterance_3.wav,3,3,0.521401,0.297078,0.580893
3,dialogue_3_utterance_4.wav,3,4,0.574994,0.229676,0.622504
4,dialogue_3_utterance_5.wav,3,5,0.515639,0.404277,0.562252
5,dialogue_3_utterance_6.wav,3,6,0.735682,0.760016,0.740866


## 2) โหลด text VAD

In [3]:
text_csv = "../../own_script/dialogue_3/dialogue_3_vad_text.csv"
df_text = pd.read_csv(text_csv)

df_text


,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n
0,1,I've been feeling really overwhelmed lately. M...,2.097536,3.681398,2.381162,-0.451232,0.340699,-0.309419
1,2,"When I feel the palpitations, my heart starts ...",2.330700,3.645309,2.423089,-0.334650,0.322655,-0.288456
2,3,"Well, the last time I went to the doctor, they...",2.717745,3.235296,2.625666,-0.141128,0.117648,-0.187167
3,4,I guess if I could really believe that my hear...,2.622219,3.526547,2.657344,-0.188891,0.263273,-0.171328
4,5,"I think trying those techniques could help, bu...",2.975230,3.375428,3.043380,-0.012385,0.187714,0.021690
5,6,That sounds like a good idea! I think having a...,3.274841,3.524532,3.352258,0.137421,0.262266,0.176129


## 3) สเกล speech logits ไปช่วง [-1,1] --- Comment ที่จะใช้

### สเกลโดยการใช้ mapping คงที่ เพื่อคงค่า absoloute ไว้

In [4]:
cols = ["arousal", "dominance", "valence"]

for c in cols:
    x = df_speech[c].values  # สมมติค่านี้ ~ [0, 1] อยู่แล้ว
    x = np.clip(x, 0.0, 1.0)  # กันกรณีหลุดนิดหน่อย
    df_speech[c + "_scaled"] = 2 * x - 1   # map 0..1 -> -1..1

df_speech[["arousal_scaled", "dominance_scaled", "valence_scaled"]].describe()


,arousal_scaled,dominance_scaled,valence_scaled
count,6.000000,6.000000,6.000000
mean,0.210906,0.295975,-0.306692
std,0.200619,0.160334,0.435444
min,0.031278,0.124503,-0.637436
25%,0.060711,0.182591,-0.573768
50%,0.132213,0.252563,-0.473247
75%,0.379174,0.426329,-0.245046
max,0.471364,0.502704,0.520031


#### Check preserved order after min/max scaling for speech_df

In [5]:
cols_speech = ["arousal", "dominance", "valence"]

def check_order_preserved(df, col, scaled_col):
    idx = np.argsort(df[col].values)
    x = df[col].values[idx]
    y = df[scaled_col].values[idx]
    return np.all(y[1:] >= y[:-1])

print("=== speech ===")
for c in cols_speech:
    ok = check_order_preserved(df_speech, c, c + "_scaled")
    print(f"{c}: order preserved? {ok}")

=== speech ===
arousal: order preserved? True
dominance: order preserved? True
valence: order preserved? True


In [6]:
df_speech

,filename,dialogue_id,utterance_id,arousal,valence,dominance,arousal_scaled,dominance_scaled,valence_scaled
0,dialogue_3_utterance_1.wav,3,1,0.557219,0.181282,0.630059,0.114438,0.260118,-0.637436
1,dialogue_3_utterance_2.wav,3,2,0.727785,0.207596,0.751352,0.455570,0.502704,-0.584808
2,dialogue_3_utterance_3.wav,3,3,0.521401,0.297078,0.580893,0.042801,0.161786,-0.405845
3,dialogue_3_utterance_4.wav,3,4,0.574994,0.229676,0.622504,0.149988,0.245007,-0.540649
4,dialogue_3_utterance_5.wav,3,5,0.515639,0.404277,0.562252,0.031278,0.124503,-0.191446
5,dialogue_3_utterance_6.wav,3,6,0.735682,0.760016,0.740866,0.471364,0.481733,0.520031


## 4) สเกล text VAD ไปช่วง [-1,1]

### สเกลโดยการใช้ mapping คงที่ เพื่อคงค่า absoloute ไว้

In [7]:
cols_text = ["valence_text", "arousal_text", "dominance_text"]

for c in cols_text:
    x = df_text[c].values          # raw ในช่วง ~[1, 5]
    x = np.clip(x, 1.0, 5.0)       # กันหลุดนอกสเกลเล็กน้อย
    x_01 = (x - 1.0) / (5.0 - 1.0) # map 1..5 -> 0..1
    df_text[c.replace("_text", "_scaled")] = x_01 * 2 - 1  # 0..1 -> -1..1



df_text[["valence_scaled", "arousal_scaled", "dominance_scaled"]].describe()


,valence_scaled,arousal_scaled,dominance_scaled
count,6.000000,6.000000,6.000000
mean,-0.165144,0.249043,-0.126425
std,0.212885,0.083932,0.189168
min,-0.451232,0.117648,-0.309419
25%,-0.298210,0.206352,-0.263133
50%,-0.165009,0.262770,-0.179248
75%,-0.044571,0.307809,-0.026564
max,0.137420,0.340699,0.176129


#### Check preserved order after min/max scaling

In [8]:
import numpy as np

cols_text = ["valence_text", "arousal_text", "dominance_text"]

def check_order_preserved(df, col, scaled_col):
    # sort ตามค่าดั้งเดิม
    idx = np.argsort(df[col].values)
    x = df[col].values[idx]
    y = df[scaled_col].values[idx]

    # x ต้อง non-decreasing อยู่แล้วเพราะ sort; เช็คว่า y ก็ non-decreasing
    return np.all(y[1:] >= y[:-1])

print("=== text ===")
for c in cols_text:
    base_col   = c
    scaled_col = c.replace("_text", "_scaled")
    ok = check_order_preserved(df_text, base_col, scaled_col)
    print(f"{base_col}: order preserved? {ok}")


=== text ===
valence_text: order preserved? True
arousal_text: order preserved? True
dominance_text: order preserved? True


## 5) ตั้งชื่อให้ไม่ชนกัน แล้ว merge

In [9]:
import pandas as pd
import numpy as np

# 1) เตรียมให้ utterance_id ตรงกัน (dialogue 1)
speech = df_speech[df_speech["dialogue_id"] == 3].sort_values("utterance_id").reset_index(drop=True)
text   = df_text.sort_values("utterance_id").reset_index(drop=True)

assert (speech["utterance_id"].to_numpy() == text["utterance_id"].to_numpy()).all()

# 2) ดึงเฉพาะ valence + arousal ที่สเกลแล้ว [-1,1]
df_dissonance = pd.DataFrame({
    "utterance_id": speech["utterance_id"],
    "aro_s": speech["arousal_scaled"].to_numpy(),
    "val_s": speech["valence_scaled"].to_numpy(),
    "aro_t": text["arousal_scaled"].to_numpy(),
    "val_t": text["valence_scaled"].to_numpy(),
})


## 6) คำนวณ dissonance (เวอร์ชันที่กันบั๊ก dtype แล้ว)

In [10]:
# 3) delta ต่อมิติ
df_dissonance["delta_arousal"] = (df_dissonance["aro_s"] - df_dissonance["aro_t"]).abs()
df_dissonance["delta_valence"] = (df_dissonance["val_s"] - df_dissonance["val_t"]).abs()

# 4) threshold บน [-1,1]
ARO_THR = 0.5
VAL_THR = 0.5

df_dissonance["dissonant_arousal"] = df_dissonance["delta_arousal"] > ARO_THR
df_dissonance["dissonant_valence"] = df_dissonance["delta_valence"] > VAL_THR
df_dissonance["dissonant_any"]     = (
    df_dissonance["dissonant_arousal"] | df_dissonance["dissonant_valence"]
)

# # 5) optional: score เดียว (L2 จาก 2 มิติ)
# df_dissonance["dissonance_l2"] = np.sqrt(
#     df_dissonance["delta_arousal"]**2 + df_dissonance["delta_valence"]**2
# )

In [11]:
df_dissonance

,utterance_id,aro_s,val_s,aro_t,val_t,delta_arousal,delta_valence,dissonant_arousal,dissonant_valence,dissonant_any
0,1,0.114438,-0.637436,0.340699,-0.451232,0.226261,0.186204,False,False,False
1,2,0.455570,-0.584808,0.322655,-0.334650,0.132915,0.250158,False,False,False
2,3,0.042801,-0.405845,0.117648,-0.141128,0.074847,0.264717,False,False,False
3,4,0.149988,-0.540649,0.263273,-0.188891,0.113286,0.351758,False,False,False
4,5,0.031278,-0.191446,0.187714,-0.012385,0.156436,0.179061,False,False,False
5,6,0.471364,0.520031,0.262266,0.137420,0.209098,0.382611,False,False,False


In [12]:
df_speech

,filename,dialogue_id,utterance_id,arousal,valence,dominance,arousal_scaled,dominance_scaled,valence_scaled
0,dialogue_3_utterance_1.wav,3,1,0.557219,0.181282,0.630059,0.114438,0.260118,-0.637436
1,dialogue_3_utterance_2.wav,3,2,0.727785,0.207596,0.751352,0.455570,0.502704,-0.584808
2,dialogue_3_utterance_3.wav,3,3,0.521401,0.297078,0.580893,0.042801,0.161786,-0.405845
3,dialogue_3_utterance_4.wav,3,4,0.574994,0.229676,0.622504,0.149988,0.245007,-0.540649
4,dialogue_3_utterance_5.wav,3,5,0.515639,0.404277,0.562252,0.031278,0.124503,-0.191446
5,dialogue_3_utterance_6.wav,3,6,0.735682,0.760016,0.740866,0.471364,0.481733,0.520031


In [13]:
df_text

,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n,valence_scaled,arousal_scaled,dominance_scaled
0,1,I've been feeling really overwhelmed lately. M...,2.097536,3.681398,2.381162,-0.451232,0.340699,-0.309419,-0.451232,0.340699,-0.309419
1,2,"When I feel the palpitations, my heart starts ...",2.330700,3.645309,2.423089,-0.334650,0.322655,-0.288456,-0.334650,0.322655,-0.288456
2,3,"Well, the last time I went to the doctor, they...",2.717745,3.235296,2.625666,-0.141128,0.117648,-0.187167,-0.141128,0.117648,-0.187167
3,4,I guess if I could really believe that my hear...,2.622219,3.526547,2.657344,-0.188891,0.263273,-0.171328,-0.188891,0.263273,-0.171328
4,5,"I think trying those techniques could help, bu...",2.975230,3.375428,3.043380,-0.012385,0.187714,0.021690,-0.012385,0.187714,0.021690
5,6,That sounds like a good idea! I think having a...,3.274841,3.524532,3.352258,0.137421,0.262266,0.176129,0.137420,0.262266,0.176129


In [14]:
output_path = "../../own_script/dialogue_3/dialogue_3_dissonance.csv"
df_dissonance.to_csv(output_path, index=False)
print("Saved:", output_path)


Saved: ../../own_script/dialogue_3/dialogue_3_dissonance.csv
